### The Controller
(Ohne Mousevents).

In [ ]:
import widget_helpers as W
from IPython.display import display


class Controller:
    def __init__(self, game, view, callbacks, key_handler=None, debug=True):
        self.game = game
        self.view = view
        self.callbacks = callbacks
        self.key_handler = key_handler
        self.debug = debug

        if debug:
            self.out = W.get_out()
            self.on_key_down = self.out.capture(clear_output=True)(self.on_key_down)
            self.log = self.out.capture(clear_output=True)(self.log)

        self._state = {}
        self.register_callbacks()

    def log(self, msg):
        if self.debug:
            print(msg)

    def register_callbacks(self):
        self.view.mcanvas.on_key_down(self.on_key_down)


    def on_key_down(self, key, *flags):
        self.log(f'Controller: Key {key!r} got pressed')

        if key in self.callbacks:
            self.log(f'calling {self.callbacks[key].__name__}()')

            self.callbacks[key]()

        if self.key_handler:
            msg = f'calling key_handler(controller, key={key!r}, state={self._state})'
            self.log(msg)

            self.key_handler(self, key, self._state)

            self.log(f'{msg}\nstate is now {self._state}')

    def display(self):
        self.view.display()
        if self.debug:
            display(self.out)

    def _ipython_display_(self):
        self.display()

In [ ]:
from model_view_controller import notify, Observable, BaseView
from gridhelper import GridHelper


class Game(Observable):
    def __init__(self):
        self.ncol = 4
        self.nrow = 4

    @notify  # ruft _notify(<methodenname>, <zurueckgeg. Argumente>) auf
    def new_game(self):
        self.player_pos = (0, 0)

    def is_inside(self, col, row):
        return 0 <= col < self.ncol and 0 <= row < self.nrow

    @notify
    def move(self, dx, dy):
        x, y = self.player_pos
        new_pos = (x+dx, y+dy)
        if self.is_inside(*new_pos):
            self.player_pos = new_pos
            return new_pos


class View(BaseView):
    def __init__(self, game, width=100, height=100, nlayers=1, debug=True):
        super().__init__(game, width, height, nlayers, debug)

        self.gridhelper = GridHelper(10, 10, 20, 20, 4, 4)
        self.gridhelper.draw_grid(self.canvas, line_width=2, color='blue')

        self.log('Drawing the Grid')

    def update(self, event, data):
        self.log(f'running update(event={event}, data={data})')
        self.canvas.clear()
        self.gridhelper.draw_grid(self.canvas, line_width=2, color='blue')
        self.gridhelper.fill_circle(self.canvas, self.game.player_pos, color='red')

In [ ]:
game = Game()


callbacks = {
    'n': game.new_game,
    'ArrowRight': lambda: game.move(1, 0),
    'ArrowLeft': lambda: game.move(-1, 0),
    'ArrowUp': lambda: game.move(0, -1),
    'ArrowDown': lambda: game.move(0, 1),
    }


def key_handler(self, key, state):
    tt = {'Right': (1, 0), 'Left': (-1, 0), 'Up': (0, -1), 'Down': (0, 1)}
    if key == 'Escape':
        double = state.get('double', False)
        state['double'] = not double
        return

    if key.startswith('Arrow') and state.get('double'):
        dx, dy = tt[key[5:]]
        game.move(dx, dy)


view = View(game)
controller = Controller(game, view, callbacks=callbacks, key_handler=key_handler)
controller